# Transaction Risk Monitoring Validation

This notebook validates the transaction-risk logic built on a 50,000-row
sample of the PaySim synthetic financial transaction dataset.

The analysis focuses on:
- transaction and fraud patterns;
- selection of operational risk signals;
- validation of the weighted risk score;
- review-priority assignment;
- final dashboard mart quality checks.

In [1]:
import duckdb
con = duckdb.connect("../reconciliation.duckdb")


## 1. Transaction risk exploration

Fraud occurrence is first compared across transaction types to identify
whether particular transaction categories provide a useful operational
risk signal.

In [2]:
con.sql("""
SELECT
    type,
    COUNT(*) AS transactions,
    SUM(is_fraud) AS fraud_cases,
    AVG(is_fraud) AS fraud_rate
FROM stg_transactions
GROUP BY type
ORDER BY fraud_rate DESC
""").show()

┌──────────┬──────────────┬─────────────┬──────────────────────┐
│   type   │ transactions │ fraud_cases │      fraud_rate      │
│ varchar  │    int64     │   int128    │        double        │
├──────────┼──────────────┼─────────────┼──────────────────────┤
│ TRANSFER │         4700 │          49 │ 0.010425531914893617 │
│ CASH_OUT │        13756 │          51 │ 0.003707473102646118 │
│ CASH_IN  │         8992 │           0 │                  0.0 │
│ PAYMENT  │        21912 │           0 │                  0.0 │
│ DEBIT    │          640 │           0 │                  0.0 │
└──────────┴──────────────┴─────────────┴──────────────────────┘



## 2. Large-amount threshold

The 95th percentile of transaction amount is used as a simple,
data-driven threshold for unusually large transactions.

In [3]:
con.sql("""SELECT
    PERCENTILE_CONT(0.95)
        WITHIN GROUP (ORDER BY amount) AS amount_p95
FROM stg_transactions
""").show()

┌───────────────────┐
│    amount_p95     │
│      double       │
├───────────────────┤
│ 578381.8794999995 │
└───────────────────┘



In [4]:
con.sql("""SELECT
    CASE
        WHEN amount > (
            SELECT PERCENTILE_CONT(0.95)
            WITHIN GROUP (ORDER BY amount)
            FROM stg_transactions
        )
        THEN 'Large Amount'
        ELSE 'Normal Amount'
    END AS amount_group,

    COUNT(*) AS transactions,
    SUM(is_fraud) AS fraud_cases,
    AVG(is_fraud)*100 AS fraud_rate

FROM stg_transactions

GROUP BY amount_group
""").show()

┌───────────────┬──────────────┬─────────────┬─────────────────────┐
│ amount_group  │ transactions │ fraud_cases │     fraud_rate      │
│    varchar    │    int64     │   int128    │       double        │
├───────────────┼──────────────┼─────────────┼─────────────────────┤
│ Large Amount  │         2500 │          15 │                 0.6 │
│ Normal Amount │        47500 │          85 │ 0.17894736842105263 │
└───────────────┴──────────────┴─────────────┴─────────────────────┘



Large transaction size alone provides a weaker fraud signal than
transaction type. It is therefore retained as a secondary risk indicator
rather than being weighted equally in the final score.

## 3. Combined risk signals

The two selected operational signals are evaluated together before
constructing the final risk score.

`flag_risky_type` receives greater weight because transaction type shows
a stronger relationship with fraud in the sampled data, while
`flag_large_amount` is retained as a secondary indicator.

In [5]:
con.sql("""SELECT
    flag_large_amount,
    flag_risky_type,
    COUNT(*) AS transactions,
    SUM(is_fraud) AS fraud_cases,
    ROUND(AVG(is_fraud) * 100, 3) AS fraud_rate_pct
FROM mart_transaction_risk_scores
GROUP BY
    flag_large_amount,
    flag_risky_type
ORDER BY
    flag_risky_type,
    flag_large_amount
""").show()

┌───────────────────┬─────────────────┬──────────────┬─────────────┬────────────────┐
│ flag_large_amount │ flag_risky_type │ transactions │ fraud_cases │ fraud_rate_pct │
│      boolean      │     boolean     │    int64     │   int128    │     double     │
├───────────────────┼─────────────────┼──────────────┼─────────────┼────────────────┤
│ false             │ false           │        31440 │           0 │            0.0 │
│ true              │ false           │          104 │           0 │            0.0 │
│ false             │ true            │        16060 │          85 │          0.529 │
│ true              │ true            │         2396 │          15 │          0.626 │
└───────────────────┴─────────────────┴──────────────┴─────────────┴────────────────┘



## 4. Weighted risk score validation

The final score combines the selected signals:

- risky transaction type = 2 points
- large transaction amount = 1 point

This produces a score from 0 to 3. Fraud occurrence is then compared
across score levels to check whether higher scores correspond to
higher observed risk.

In [6]:
con.sql("""
SELECT
    risk_score,
    COUNT(*) AS transactions,
    SUM(is_fraud) AS fraud_cases,
    ROUND(AVG(is_fraud) * 100, 3) AS fraud_rate_pct
FROM mart_transaction_risk_scores
GROUP BY risk_score
ORDER BY risk_score
""").show()

┌────────────┬──────────────┬─────────────┬────────────────┐
│ risk_score │ transactions │ fraud_cases │ fraud_rate_pct │
│   int32    │    int64     │   int128    │     double     │
├────────────┼──────────────┼─────────────┼────────────────┤
│          0 │        31440 │           0 │            0.0 │
│          1 │          104 │           0 │            0.0 │
│          2 │        16060 │          85 │          0.529 │
│          3 │         2396 │          15 │          0.626 │
└────────────┴──────────────┴─────────────┴────────────────┘



## 5. Review-priority assignment

Risk scores are translated into operational review categories so that
the dashboard can support transaction triage rather than display a raw
score alone.

In [7]:
con.sql("""
SELECT
    review_priority,
    risk_score,
    COUNT(*) AS transactions
FROM mart_transaction_risk_dashboard
GROUP BY review_priority, risk_score
ORDER BY risk_score
""").show()

┌─────────────────┬────────────┬──────────────┐
│ review_priority │ risk_score │ transactions │
│     varchar     │   int32    │    int64     │
├─────────────────┼────────────┼──────────────┤
│ No Risk Signals │          0 │        31440 │
│ Large Amount    │          1 │          104 │
│ Review          │          2 │        16060 │
│ High Priority   │          3 │         2396 │
└─────────────────┴────────────┴──────────────┘



## 6. Dashboard mart validation

Before export, the final mart is checked for transaction-level grain
and consistency between the component risk flags and the calculated
risk score.

In [8]:
con.sql("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT (
        step,
        type,
        amount,
        sender_id,
        receiver_id
    )) AS unique_transactions
FROM mart_transaction_risk_dashboard
""").show()

┌───────┬─────────────────────┐
│ rows  │ unique_transactions │
│ int64 │        int64        │
├───────┼─────────────────────┤
│ 50000 │               50000 │
└───────┴─────────────────────┘



In [9]:
con.sql("""
SELECT COUNT(*) AS invalid_scores
FROM mart_transaction_risk_dashboard
WHERE risk_score !=
      (flag_risky_type::INT * 2 + flag_large_amount::INT)
""").show()

┌────────────────┐
│ invalid_scores │
│     int64      │
├────────────────┤
│              0 │
└────────────────┘



The final mart contains 50,000 transaction-level records with no
inconsistencies between the component risk flags and the calculated
risk score.

## 7. Tableau export

The validated transaction-level mart is exported as the data source
for the interactive Transaction Risk Monitoring dashboard.

In [10]:
con.sql("""
COPY mart_transaction_risk_dashboard
TO '../dashboard/transaction_risk_dashboard.csv'
(HEADER, DELIMITER ',')
""")